# 향후 희망 여가활동 소득 변수 실험

## 분석 환경 및 데이터 경로 설정
- 기존 모델링 구조를 유지하고 소득/가구원수 변수를 추가한 실험을 수행함.
- 기존 산출물은 덮어쓰지 않고 income_exp 결과만 별도로 저장함.

In [ ]:

import pathlib
import pandas as pd
import numpy as np

from scipy.optimize import minimize
from scipy.special import logsumexp
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    log_loss,
    f1_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

BASE_PATH = pathlib.Path().resolve()
PROJECT_PATH = BASE_PATH

while PROJECT_PATH.name != "oracle_mnc_project" and PROJECT_PATH.parent != PROJECT_PATH:
    PROJECT_PATH = PROJECT_PATH.parent

PREFERENCE_PATH = PROJECT_PATH / "notebooks" / "preference"
SOURCE_PATH = PREFERENCE_PATH / "data" / "source"
PROCESSED_PATH = PREFERENCE_PATH / "data" / "processed"
SATISFACTION_PATH = PROCESSED_PATH / "satisfaction"
INCOME_EXP_PATH = PROCESSED_PATH / "income_exp"
INCOME_EXP_PATH.mkdir(parents=True, exist_ok=True)

MAPPING_PATH = SATISFACTION_PATH / "ml_activity_category_mapping.csv"

print("PROJECT_PATH:", PROJECT_PATH)
print("MAPPING_PATH 존재:", MAPPING_PATH.exists())
print("INCOME_EXP_PATH:", INCOME_EXP_PATH)

OUTPUT_PREFIX = "preference"


## 데이터 불러오기 및 소득 변수 설계
- 문화누리 중분류 매핑표의 학습 사용 가능 분류만 타깃으로 사용함.
- 소득 변수는 전 연도 공통으로 존재하는 가구소득 생성값과 동거가구원수 생성값을 사용함.

In [ ]:

sex_map = {
    1: "남성",
    2: "여성",
}

age_map = {
    1: "15-19세",
    2: "20대",
    3: "30대",
    4: "40대",
    5: "50대",
    6: "60대",
    7: "70세 이상",
}

sido_map = {
    1: "서울",
    2: "부산",
    3: "대구",
    4: "인천",
    5: "광주",
    6: "대전",
    7: "울산",
    8: "세종",
    9: "경기",
    10: "강원",
    11: "충북",
    12: "충남",
    13: "전북",
    14: "전남",
    15: "경북",
    16: "경남",
    17: "제주",
}

region_size_map = {
    1: "대도시",
    2: "중소도시",
    3: "읍면지역",
}

income_map = {
    1: "100만원 미만",
    2: "100-200만원",
    3: "200-300만원",
    4: "300-400만원",
    5: "400-500만원",
    6: "500-600만원",
    7: "600만원 이상",
    9: "모름/무응답",
}

household_size_map = {
    1: "1인",
    2: "2인",
    3: "3인 이상",
}

income_order = [
    "100만원 미만",
    "100-200만원",
    "200-300만원",
    "300-400만원",
    "400-500만원",
    "500-600만원",
    "600만원 이상",
    "모름/무응답",
]

household_order = ["1인", "2인", "3인 이상"]


def add_common_labels(data):
    result = data.copy()
    result["성별_라벨"] = result["성별"].map(sex_map).fillna("미상")
    result["연령대"] = result["연령"].map(age_map).fillna("미상")
    result["시도"] = result["17개 시도"].map(sido_map).fillna("미상")
    result["지역규모_라벨"] = result["지역규모"].map(region_size_map).fillna("미상")
    result["조사년도_라벨"] = result["조사년도"].astype(str)
    
    income_source = "가구소득_생성" if "가구소득_생성" in result.columns else "가구소득"
    result["가구소득_라벨"] = (
        pd.to_numeric(result[income_source], errors="coerce")
        .map(income_map)
        .fillna("모름/무응답")
    )
    
    if "동거가구원수_생성" in result.columns:
        result["동거가구원수_라벨"] = (
            pd.to_numeric(result["동거가구원수_생성"], errors="coerce")
            .map(household_size_map)
            .fillna("미상")
        )
    else:
        result["동거가구원수_라벨"] = "미상"
    
    return result


In [ ]:

SURVEY_PATH = SOURCE_PATH / "leisure_activity_survey_2021_2025_selected_columns_enriched.csv"

survey = pd.read_csv(SURVEY_PATH, encoding="utf-8-sig", low_memory=False)
activity_mapping = pd.read_csv(MAPPING_PATH, encoding="utf-8-sig")

activity_mapping["활동코드"] = pd.to_numeric(
    activity_mapping["활동코드"],
    errors="coerce",
).astype("Int64")

future_rank_cols_raw = [
    "향후 희망하는 여가활동 1순위",
    "향후 희망하는 여가활동 2순위",
    "향후 희망하는 여가활동 3순위",
]

valid_categories = (
    activity_mapping
    .loc[activity_mapping["학습타깃사용여부"], "중분류"]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

survey_2425 = survey[survey["조사년도"].isin([2024, 2025])].copy()

feature_cols = [
    "응답자_ID",
    "조사년도",
    "성별",
    "연령",
    "17개 시도",
    "지역규모",
    "최종가중치",
    "학력",
    "가구소득",
    "장애여부",
    "동거가구원수_생성",
    "전체동거가구원수_원문항",
    "월평균가구소득_원코드",
    "가구소득_생성",
]

mapping_info = (
    activity_mapping
    .set_index("활동코드")[["여가활동명", "중분류", "학습타깃사용여부"]]
    .to_dict("index")
)

future_base = survey_2425[feature_cols + future_rank_cols_raw].copy().reset_index(drop=True)

rank_rows = []

for idx, row in future_base.iterrows():
    valid_rank = []
    exclude_count = 0
    duplicate_count = 0
    raw_codes = []
    raw_categories = []
    
    for col in future_rank_cols_raw:
        code = pd.to_numeric(row[col], errors="coerce")
        
        if pd.isna(code):
            raw_codes.append(np.nan)
            raw_categories.append(np.nan)
            continue
        
        code = int(code)
        raw_codes.append(code)
        info = mapping_info.get(code)
        
        if info is None:
            raw_categories.append(np.nan)
            exclude_count += 1
            continue
        
        category = info["중분류"]
        raw_categories.append(category)
        
        if not bool(info["학습타깃사용여부"]):
            exclude_count += 1
            continue
        
        if category in valid_rank:
            duplicate_count += 1
            continue
        
        valid_rank.append(category)
    
    temp = {
        "응답자_ID": row["응답자_ID"],
        "향후희망_원코드_1순위": raw_codes[0],
        "향후희망_원코드_2순위": raw_codes[1],
        "향후희망_원코드_3순위": raw_codes[2],
        "향후희망_원중분류_1순위": raw_categories[0],
        "향후희망_원중분류_2순위": raw_categories[1],
        "향후희망_원중분류_3순위": raw_categories[2],
        "향후희망_유효순위수": len(valid_rank),
        "제외분류_제거수": exclude_count,
        "중복중분류_제거수": duplicate_count,
    }
    
    for rank in range(3):
        temp[f"향후희망_유효중분류_{rank + 1}순위"] = valid_rank[rank] if rank < len(valid_rank) else np.nan
    
    rank_rows.append(temp)

future_rank_info = pd.DataFrame(rank_rows)
future_rank_base = future_base.drop(columns=future_rank_cols_raw).merge(
    future_rank_info,
    on="응답자_ID",
    how="left",
)

rank_cols = [
    "향후희망_유효중분류_1순위",
    "향후희망_유효중분류_2순위",
    "향후희망_유효중분류_3순위",
]

model_df = future_rank_base[future_rank_base[rank_cols[0]].notna()].copy()
model_df = add_common_labels(model_df)

print("survey 구조:", survey.shape)
print("future_rank_base 구조:", future_rank_base.shape)
print("모델링 대상 구조:", model_df.shape)
print("중분류 후보군:", valid_categories)
print("학습 가능한 응답자 수:", model_df[rank_cols[0]].notna().sum())
print("유효순위수 분포")
print(model_df["향후희망_유효순위수"].value_counts(dropna=False).sort_index())
display(model_df.head())


## 소득 변수 품질 점검 및 데이터 분할
- 소득/가구원수 변수의 결측과 분포를 확인함.
- 조사년도와 1순위 중분류를 함께 고려해 train, valid, test를 층화 분할함.

In [ ]:

income_missing = (
    model_df[
        [
            "가구소득_라벨",
            "동거가구원수_라벨",
            "가구소득_생성",
            "동거가구원수_생성",
            "월평균가구소득_원코드",
            "전체동거가구원수_원문항",
        ]
    ]
    .isna()
    .sum()
    .reset_index(name="결측치")
    .rename(columns={"index": "칼럼명"})
)
income_missing["결측률"] = income_missing["결측치"] / len(model_df)

print("소득/가구원수 변수 결측 점검")
display(income_missing)

print("가구소득 분포")
display(model_df["가구소득_라벨"].value_counts(dropna=False).reindex(income_order).dropna().reset_index().rename(columns={"index": "가구소득", "가구소득_라벨": "응답자수"}))

print("동거가구원수 분포")
display(model_df["동거가구원수_라벨"].value_counts(dropna=False).reindex(household_order).dropna().reset_index().rename(columns={"index": "동거가구원수", "동거가구원수_라벨": "응답자수"}))

print("가구소득별 1순위 중분류 분포")
income_rank1_dist = pd.crosstab(
    model_df["가구소득_라벨"],
    model_df[rank_cols[0]],
    normalize="index",
).reindex(income_order)
display(income_rank1_dist)

split_key = (
    model_df["조사년도_라벨"]
    + "_"
    + model_df[rank_cols[0]]
)

train_valid_idx, test_idx = train_test_split(
    model_df.index,
    test_size=0.15,
    random_state=42,
    stratify=split_key,
)

train_valid_df = model_df.loc[train_valid_idx].copy()
train_valid_key = (
    train_valid_df["조사년도_라벨"]
    + "_"
    + train_valid_df[rank_cols[0]]
)

train_idx, valid_idx = train_test_split(
    train_valid_df.index,
    test_size=0.15 / 0.85,
    random_state=42,
    stratify=train_valid_key,
)

train_df = model_df.loc[train_idx].copy()
valid_df = model_df.loc[valid_idx].copy()
test_df = model_df.loc[test_idx].copy()

split_summary = pd.DataFrame({
    "데이터": ["train", "valid", "test"],
    "응답자수": [len(train_df), len(valid_df), len(test_df)],
    "비율": [len(train_df) / len(model_df), len(valid_df) / len(model_df), len(test_df) / len(model_df)],
})

display(split_summary)

split_label_df = pd.concat([
    train_df.assign(split="train"),
    valid_df.assign(split="valid"),
    test_df.assign(split="test"),
])

print("1순위 중분류 분포")
display(pd.crosstab(
    split_label_df[rank_cols[0]],
    split_label_df["split"],
    normalize="columns",
))


## 모델 함수 정의
- Rank-Ordered Logit과 다항 로지스틱을 동일한 split에서 학습함.
- baseline 입력 변수와 income 입력 변수를 비교함.

In [ ]:

baseline_feature_cols = [
    "성별_라벨",
    "연령대",
    "시도",
    "지역규모_라벨",
    "조사년도_라벨",
]

income_feature_cols = baseline_feature_cols + [
    "가구소득_라벨",
    "동거가구원수_라벨",
]

reference_category = "영상"
l2_alpha = 1e-4

category_to_idx = {
    category: idx
    for idx, category in enumerate(valid_categories)
}

idx_to_category = {
    idx: category
    for category, idx in category_to_idx.items()
}

reference_idx = category_to_idx[reference_category]
nonref_idx = [
    idx for idx in range(len(valid_categories))
    if idx != reference_idx
]


def make_feature_matrix(data, feature_cols):
    feature_df = pd.get_dummies(
        data[feature_cols],
        drop_first=True,
        dtype=float,
    )
    feature_df.insert(0, "상수", 1.0)
    return feature_df


def make_model_arrays(data, feature_df):
    X = feature_df.loc[data.index].to_numpy(dtype=float)
    y = np.full((len(data), 3), -1, dtype=int)
    
    for rank, col in enumerate(rank_cols):
        y[:, rank] = (
            data[col]
            .map(category_to_idx)
            .fillna(-1)
            .astype(int)
            .to_numpy()
        )
    
    sample_weight = data["최종가중치"].to_numpy(dtype=float)
    sample_weight = sample_weight / np.nanmean(sample_weight)
    
    return X, y, sample_weight


def fit_rol_model(X_train, y_train, weight_train, maxiter=500):
    K = len(valid_categories)
    D = X_train.shape[1]
    
    choice_weight_sum = sum(
        weight_train[y_train[:, rank] >= 0].sum()
        for rank in range(3)
    )
    
    def unpack_params(params):
        W_nonref = params.reshape(D, K - 1)
        W = np.zeros((D, K), dtype=float)
        W[:, nonref_idx] = W_nonref
        return W
    
    def loss_grad(params):
        W = unpack_params(params)
        scores = X_train @ W
        grad_scores = np.zeros_like(scores)
        loss = 0.0
        
        for rank in range(3):
            valid_mask = y_train[:, rank] >= 0
            
            if valid_mask.sum() == 0:
                continue
            
            valid_idx = np.where(valid_mask)[0]
            chosen = y_train[valid_idx, rank]
            
            masked_scores = scores[valid_idx].copy()
            remain_mask = np.ones_like(masked_scores, dtype=bool)
            
            for prev_rank in range(rank):
                prev_chosen = y_train[valid_idx, prev_rank]
                remain_mask[np.arange(len(valid_idx)), prev_chosen] = False
            
            masked_scores[~remain_mask] = -np.inf
            log_den = logsumexp(masked_scores, axis=1)
            probs = np.exp(masked_scores - log_den[:, None])
            probs[~remain_mask] = 0
            
            row_grad = probs
            row_grad[np.arange(len(valid_idx)), chosen] -= 1
            
            row_weight = weight_train[valid_idx]
            loss += np.sum(row_weight * (log_den - scores[valid_idx, chosen]))
            grad_scores[valid_idx] += row_grad * row_weight[:, None]
        
        grad_full = X_train.T @ grad_scores
        grad_nonref = grad_full[:, nonref_idx]
        params_nonref = W[:, nonref_idx].ravel()
        
        loss = loss / choice_weight_sum + 0.5 * l2_alpha * np.sum(params_nonref ** 2)
        grad = (grad_nonref / choice_weight_sum).ravel() + l2_alpha * params_nonref
        
        return loss, grad
    
    init_params = np.zeros(D * (K - 1), dtype=float)
    
    result = minimize(
        fun=lambda params: loss_grad(params),
        x0=init_params,
        jac=True,
        method="L-BFGS-B",
        options={"maxiter": maxiter},
    )
    
    return unpack_params(result.x), result


def predict_prob(X, W):
    scores = X @ W
    scores = scores - scores.max(axis=1, keepdims=True)
    exp_scores = np.exp(scores)
    return exp_scores / exp_scores.sum(axis=1, keepdims=True)


def rol_log_loss(X, y, W, sample_weight=None):
    if sample_weight is None:
        sample_weight = np.ones(X.shape[0])
    
    scores = X @ W
    total_loss = 0.0
    total_weight = 0.0
    
    for rank in range(3):
        valid_mask = y[:, rank] >= 0
        
        if valid_mask.sum() == 0:
            continue
        
        valid_idx = np.where(valid_mask)[0]
        chosen = y[valid_idx, rank]
        
        masked_scores = scores[valid_idx].copy()
        remain_mask = np.ones_like(masked_scores, dtype=bool)
        
        for prev_rank in range(rank):
            prev_chosen = y[valid_idx, prev_rank]
            remain_mask[np.arange(len(valid_idx)), prev_chosen] = False
        
        masked_scores[~remain_mask] = -np.inf
        log_den = logsumexp(masked_scores, axis=1)
        
        row_weight = sample_weight[valid_idx]
        total_loss += np.sum(row_weight * (log_den - scores[valid_idx, chosen]))
        total_weight += row_weight.sum()
    
    return total_loss / total_weight


def ndcg_at_3(prob, y):
    pred_order = np.argsort(-prob, axis=1)[:, :3]
    ndcg_list = []
    
    for i in range(len(y)):
        relevance = {}
        
        for rank in range(3):
            if y[i, rank] >= 0:
                relevance[y[i, rank]] = 3 - rank
        
        if len(relevance) == 0:
            continue
        
        dcg = 0.0
        
        for position, category_idx in enumerate(pred_order[i], start=1):
            rel = relevance.get(category_idx, 0)
            dcg += (2 ** rel - 1) / np.log2(position + 1)
        
        ideal_relevance = sorted(relevance.values(), reverse=True)[:3]
        idcg = sum(
            (2 ** rel - 1) / np.log2(position + 1)
            for position, rel in enumerate(ideal_relevance, start=1)
        )
        
        ndcg_list.append(dcg / idcg if idcg > 0 else np.nan)
    
    return np.nanmean(ndcg_list)


def evaluate_rank_model(data_name, X, y, W, sample_weight=None):
    prob = predict_prob(X, W)
    pred_order = np.argsort(-prob, axis=1)
    rank1 = y[:, 0]
    
    top1_accuracy = np.mean(pred_order[:, 0] == rank1)
    top3_hit_rate = np.mean([
        rank1[i] in pred_order[i, :3]
        for i in range(len(rank1))
    ])
    mrr = np.mean([
        1 / (np.where(pred_order[i] == rank1[i])[0][0] + 1)
        for i in range(len(rank1))
    ])
    
    return {
        "데이터": data_name,
        "Top1_Accuracy": top1_accuracy,
        "Top3_HitRate": top3_hit_rate,
        "MRR": mrr,
        "NDCG@3": ndcg_at_3(prob, y),
        "ROL_LogLoss": rol_log_loss(X, y, W, sample_weight),
    }


def align_mnl_prob(model, X_mnl):
    raw_prob = model.predict_proba(X_mnl)
    prob = np.zeros((X_mnl.shape[0], len(valid_categories)))
    class_to_col = {
        category: idx
        for idx, category in enumerate(model.classes_)
    }
    
    for j, category in enumerate(valid_categories):
        if category in class_to_col:
            prob[:, j] = raw_prob[:, class_to_col[category]]
    
    row_sum = prob.sum(axis=1, keepdims=True)
    prob = np.divide(
        prob,
        row_sum,
        out=np.zeros_like(prob),
        where=row_sum > 0,
    )
    
    return prob


def evaluate_prob_rank_model(data_name, prob, y, y_label):
    pred_order = np.argsort(-prob, axis=1)
    rank1 = y[:, 0]
    
    top1_accuracy = np.mean(pred_order[:, 0] == rank1)
    top3_hit_rate = np.mean([
        rank1[i] in pred_order[i, :3]
        for i in range(len(rank1))
    ])
    mrr = np.mean([
        1 / (np.where(pred_order[i] == rank1[i])[0][0] + 1)
        for i in range(len(rank1))
    ])
    
    return {
        "데이터": data_name,
        "Top1_Accuracy": top1_accuracy,
        "Top3_HitRate": top3_hit_rate,
        "MRR": mrr,
        "NDCG@3": ndcg_at_3(prob, y),
        "Rank1_LogLoss": log_loss(y_label, prob, labels=valid_categories),
    }


def make_classification_metric(model_name, experiment_name, split_name, prob, y_true):
    pred_idx = np.argmax(prob, axis=1)
    pred_label = np.array([idx_to_category[idx] for idx in pred_idx])
    
    return {
        "실험": experiment_name,
        "모델": model_name,
        "데이터": split_name,
        "Macro_F1": f1_score(
            y_true,
            pred_label,
            labels=valid_categories,
            average="macro",
            zero_division=0,
        ),
        "Weighted_F1": f1_score(
            y_true,
            pred_label,
            labels=valid_categories,
            average="weighted",
            zero_division=0,
        ),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, pred_label),
        "Rank1_LogLoss": log_loss(
            y_true,
            prob,
            labels=valid_categories,
        ),
    }


def make_category_metric(model_name, experiment_name, split_name, prob, y_true):
    pred_idx = np.argmax(prob, axis=1)
    pred_label = np.array([idx_to_category[idx] for idx in pred_idx])
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        pred_label,
        labels=valid_categories,
        zero_division=0,
    )
    pred_count = (
        pd.Series(pred_label)
        .value_counts()
        .reindex(valid_categories, fill_value=0)
        .to_numpy()
    )
    
    return pd.DataFrame({
        "실험": experiment_name,
        "모델": model_name,
        "데이터": split_name,
        "중분류": valid_categories,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "실제건수": support,
        "예측건수": pred_count,
    })


def make_group_rank_metric(model_name, experiment_name, split_name, prob, y, data):
    pred_order = np.argsort(-prob, axis=1)
    rank1 = y[:, 0]
    
    top1 = pred_order[:, 0] == rank1
    top3 = np.array([
        rank1[i] in pred_order[i, :3]
        for i in range(len(rank1))
    ])
    reciprocal_rank = np.array([
        1 / (np.where(pred_order[i] == rank1[i])[0][0] + 1)
        for i in range(len(rank1))
    ])
    
    ndcg_list = []
    for i in range(len(y)):
        relevance = {}
        
        for rank in range(3):
            if y[i, rank] >= 0:
                relevance[y[i, rank]] = 3 - rank
        
        dcg = 0.0
        
        for position, category_idx in enumerate(pred_order[i, :3], start=1):
            rel = relevance.get(category_idx, 0)
            dcg += (2 ** rel - 1) / np.log2(position + 1)
        
        ideal_relevance = sorted(relevance.values(), reverse=True)[:3]
        idcg = sum(
            (2 ** rel - 1) / np.log2(position + 1)
            for position, rel in enumerate(ideal_relevance, start=1)
        )
        
        ndcg_list.append(dcg / idcg if idcg > 0 else np.nan)
    
    row_prob = np.clip(prob[np.arange(len(rank1)), rank1], 1e-15, 1)
    rank1_logloss = -np.log(row_prob)
    
    metric_df = data[["성별_라벨", "연령대"]].copy().reset_index(drop=True)
    metric_df["Top1"] = top1
    metric_df["Top3"] = top3
    metric_df["MRR"] = reciprocal_rank
    metric_df["NDCG@3"] = ndcg_list
    metric_df["Rank1_LogLoss"] = rank1_logloss
    
    result = (
        metric_df
        .groupby(["성별_라벨", "연령대"], as_index=False)
        .agg(
            응답자수=("Top1", "size"),
            Top1_Accuracy=("Top1", "mean"),
            Top3_HitRate=("Top3", "mean"),
            MRR=("MRR", "mean"),
            NDCG_3=("NDCG@3", "mean"),
            Rank1_LogLoss=("Rank1_LogLoss", "mean"),
        )
    )
    
    result["실험"] = experiment_name
    result["모델"] = model_name
    result["데이터"] = split_name
    
    return result[[
        "실험",
        "모델",
        "데이터",
        "성별_라벨",
        "연령대",
        "응답자수",
        "Top1_Accuracy",
        "Top3_HitRate",
        "MRR",
        "NDCG_3",
        "Rank1_LogLoss",
    ]]


def run_experiment(experiment_name, feature_cols):
    print(f"\n===== {experiment_name} =====")
    print("입력 변수:", feature_cols)
    
    feature_df = make_feature_matrix(model_df, feature_cols)
    feature_columns = feature_df.columns.tolist()
    
    X_train, y_train, weight_train = make_model_arrays(train_df, feature_df)
    X_valid, y_valid, weight_valid = make_model_arrays(valid_df, feature_df)
    X_test, y_test, weight_test = make_model_arrays(test_df, feature_df)
    
    W_hat, rol_result = fit_rol_model(X_train, y_train, weight_train)
    
    train_prior = train_df[rank_cols[0]].value_counts(normalize=True)
    prior_prob = np.array([
        train_prior.get(category, 0)
        for category in valid_categories
    ])
    prior_prob = prior_prob / prior_prob.sum()
    prior_scores = np.log(prior_prob + 1e-12)
    
    W_prior = np.zeros_like(W_hat)
    W_prior[0, :] = prior_scores - prior_scores[reference_idx]
    
    rol_performance = pd.DataFrame([
        evaluate_rank_model("train_model", X_train, y_train, W_hat, weight_train),
        evaluate_rank_model("valid_model", X_valid, y_valid, W_hat, weight_valid),
        evaluate_rank_model("test_model", X_test, y_test, W_hat, weight_test),
        evaluate_rank_model("train_prior", X_train, y_train, W_prior, weight_train),
        evaluate_rank_model("valid_prior", X_valid, y_valid, W_prior, weight_valid),
        evaluate_rank_model("test_prior", X_test, y_test, W_prior, weight_test),
    ])
    rol_performance["실험"] = experiment_name
    rol_performance["모델"] = "ROL"
    rol_performance = rol_performance.rename(columns={"ROL_LogLoss": "LogLoss"})
    rol_performance["LogLoss_기준"] = "순위"
    rol_performance["수렴여부"] = rol_result.success
    rol_performance["반복횟수"] = rol_result.nit
    
    mnl_feature_df = feature_df.drop(columns="상수").copy()
    
    X_train_mnl = mnl_feature_df.loc[train_df.index].to_numpy(dtype=float)
    X_valid_mnl = mnl_feature_df.loc[valid_df.index].to_numpy(dtype=float)
    X_test_mnl = mnl_feature_df.loc[test_df.index].to_numpy(dtype=float)
    
    y_train_mnl = train_df[rank_cols[0]].to_numpy()
    y_valid_mnl = valid_df[rank_cols[0]].to_numpy()
    y_test_mnl = test_df[rank_cols[0]].to_numpy()
    
    mnl_model = LogisticRegression(
        solver="lbfgs",
        max_iter=1000,
        C=1.0,
    )
    mnl_model.fit(
        X_train_mnl,
        y_train_mnl,
        sample_weight=weight_train,
    )
    
    mnl_prob_train = align_mnl_prob(mnl_model, X_train_mnl)
    mnl_prob_valid = align_mnl_prob(mnl_model, X_valid_mnl)
    mnl_prob_test = align_mnl_prob(mnl_model, X_test_mnl)
    
    mnl_prior_train = np.tile(prior_prob, (len(train_df), 1))
    mnl_prior_valid = np.tile(prior_prob, (len(valid_df), 1))
    mnl_prior_test = np.tile(prior_prob, (len(test_df), 1))
    
    mnl_performance = pd.DataFrame([
        evaluate_prob_rank_model("train_model", mnl_prob_train, y_train, y_train_mnl),
        evaluate_prob_rank_model("valid_model", mnl_prob_valid, y_valid, y_valid_mnl),
        evaluate_prob_rank_model("test_model", mnl_prob_test, y_test, y_test_mnl),
        evaluate_prob_rank_model("train_prior", mnl_prior_train, y_train, y_train_mnl),
        evaluate_prob_rank_model("valid_prior", mnl_prior_valid, y_valid, y_valid_mnl),
        evaluate_prob_rank_model("test_prior", mnl_prior_test, y_test, y_test_mnl),
    ])
    mnl_performance["실험"] = experiment_name
    mnl_performance["모델"] = "다항로지스틱"
    mnl_performance = mnl_performance.rename(columns={"Rank1_LogLoss": "LogLoss"})
    mnl_performance["LogLoss_기준"] = "1순위"
    mnl_performance["수렴여부"] = True
    mnl_performance["반복횟수"] = int(mnl_model.n_iter_[0])
    
    performance = pd.concat([rol_performance, mnl_performance], ignore_index=True)
    performance = performance[[
        "실험",
        "모델",
        "데이터",
        "Top1_Accuracy",
        "Top3_HitRate",
        "MRR",
        "NDCG@3",
        "LogLoss",
        "LogLoss_기준",
        "수렴여부",
        "반복횟수",
    ]]
    
    rol_prob_valid = predict_prob(X_valid, W_hat)
    rol_prob_test = predict_prob(X_test, W_hat)
    mnl_prob_valid = align_mnl_prob(mnl_model, X_valid_mnl)
    mnl_prob_test = align_mnl_prob(mnl_model, X_test_mnl)
    
    classification_metric = pd.DataFrame([
        make_classification_metric("ROL", experiment_name, "valid", rol_prob_valid, y_valid_mnl),
        make_classification_metric("ROL", experiment_name, "test", rol_prob_test, y_test_mnl),
        make_classification_metric("다항로지스틱", experiment_name, "valid", mnl_prob_valid, y_valid_mnl),
        make_classification_metric("다항로지스틱", experiment_name, "test", mnl_prob_test, y_test_mnl),
    ])
    
    category_metric = pd.concat([
        make_category_metric("ROL", experiment_name, "valid", rol_prob_valid, y_valid_mnl),
        make_category_metric("ROL", experiment_name, "test", rol_prob_test, y_test_mnl),
        make_category_metric("다항로지스틱", experiment_name, "valid", mnl_prob_valid, y_valid_mnl),
        make_category_metric("다항로지스틱", experiment_name, "test", mnl_prob_test, y_test_mnl),
    ], ignore_index=True)
    

    group_metric = pd.concat([
        make_group_rank_metric("ROL", experiment_name, "valid", rol_prob_valid, y_valid, valid_df),
        make_group_rank_metric("ROL", experiment_name, "test", rol_prob_test, y_test, test_df),
        make_group_rank_metric("다항로지스틱", experiment_name, "valid", mnl_prob_valid, y_valid, valid_df),
        make_group_rank_metric("다항로지스틱", experiment_name, "test", mnl_prob_test, y_test, test_df),
    ], ignore_index=True)
    
    print("ROL 수렴:", rol_result.success, "| 반복:", rol_result.nit)
    print("다항로지스틱 반복:", int(mnl_model.n_iter_[0]))
    
    return {
        "performance": performance,
        "classification_metric": classification_metric,
        "category_metric": category_metric,
        "group_metric": group_metric,
        "feature_columns": feature_columns,
    }


## Baseline / Income 실험 실행
- baseline은 성별, 연령대, 시도, 지역규모, 조사년도를 사용함.
- income은 baseline 변수에 가구소득과 동거가구원수를 추가함.

In [ ]:

baseline_result = run_experiment("baseline", baseline_feature_cols)
income_result = run_experiment("income", income_feature_cols)

model_performance_compare = pd.concat(
    [
        baseline_result["performance"],
        income_result["performance"],
    ],
    ignore_index=True,
).round(4)

classification_metric_compare = pd.concat(
    [
        baseline_result["classification_metric"],
        income_result["classification_metric"],
    ],
    ignore_index=True,
).round(4)

category_metric_compare = pd.concat(
    [
        baseline_result["category_metric"],
        income_result["category_metric"],
    ],
    ignore_index=True,
).round(4)

print("모델 성능 비교")
display(model_performance_compare)

print("1순위 분류 성능 비교")
display(classification_metric_compare)


gender_age_metric_compare = pd.concat(
    [
        baseline_result["group_metric"],
        income_result["group_metric"],
    ],
    ignore_index=True,
).round(4)

print("성연령별 성능 비교")
display(gender_age_metric_compare)


## 성능 변화 및 중분류별 변화 점검
- 소득 변수 추가 전후의 valid/test 성능 변화를 계산함.
- 중분류별 Recall, F1, 예측건수 변화를 확인함.

In [ ]:

target_perf = model_performance_compare[
    model_performance_compare["데이터"].isin(["valid_model", "test_model"])
].copy()

perf_delta = (
    target_perf
    .pivot_table(
        index=["모델", "데이터", "LogLoss_기준"],
        columns="실험",
        values=["Top1_Accuracy", "Top3_HitRate", "MRR", "NDCG@3", "LogLoss"],
    )
)

delta_rows = []

for model_name in target_perf["모델"].unique():
    for data_name in ["valid_model", "test_model"]:
        base_row = target_perf[
            (target_perf["실험"] == "baseline")
            & (target_perf["모델"] == model_name)
            & (target_perf["데이터"] == data_name)
        ]
        income_row = target_perf[
            (target_perf["실험"] == "income")
            & (target_perf["모델"] == model_name)
            & (target_perf["데이터"] == data_name)
        ]
        
        if len(base_row) == 0 or len(income_row) == 0:
            continue
        
        base_row = base_row.iloc[0]
        income_row = income_row.iloc[0]
        
        delta_rows.append({
            "모델": model_name,
            "데이터": data_name,
            "Top1_Accuracy_변화": income_row["Top1_Accuracy"] - base_row["Top1_Accuracy"],
            "Top3_HitRate_변화": income_row["Top3_HitRate"] - base_row["Top3_HitRate"],
            "MRR_변화": income_row["MRR"] - base_row["MRR"],
            "NDCG@3_변화": income_row["NDCG@3"] - base_row["NDCG@3"],
            "LogLoss_변화": income_row["LogLoss"] - base_row["LogLoss"],
        })

income_delta = pd.DataFrame(delta_rows).round(4)

print("소득 변수 추가 전후 성능 변화")
display(income_delta)

valid_category_delta = (
    category_metric_compare[
        (category_metric_compare["데이터"] == "valid")
        & (category_metric_compare["모델"] == "다항로지스틱")
    ]
    .pivot_table(
        index="중분류",
        columns="실험",
        values=["Recall", "F1", "예측건수"],
    )
)

category_delta_rows = []
for category in valid_categories:
    try:
        category_delta_rows.append({
            "중분류": category,
            "Recall_변화": valid_category_delta.loc[category, ("Recall", "income")] - valid_category_delta.loc[category, ("Recall", "baseline")],
            "F1_변화": valid_category_delta.loc[category, ("F1", "income")] - valid_category_delta.loc[category, ("F1", "baseline")],
            "예측건수_변화": valid_category_delta.loc[category, ("예측건수", "income")] - valid_category_delta.loc[category, ("예측건수", "baseline")],
        })
    except KeyError:
        continue

category_delta = pd.DataFrame(category_delta_rows).round(4)

print("소득 변수 추가 후 중분류별 변화 - valid / 다항로지스틱")
display(category_delta.sort_values("F1_변화", ascending=False))

low_income = "100만원 미만"
high_income = "600만원 이상"

if low_income in income_rank1_dist.index and high_income in income_rank1_dist.index:
    income_preference_gap = (
        income_rank1_dist.loc[high_income]
        - income_rank1_dist.loc[low_income]
    ).sort_values(ascending=False).reset_index()
    income_preference_gap.columns = ["중분류", "고소득-저소득_1순위비율차"]
    
    print("600만원 이상과 100만원 미만의 1순위 비율 차이")
    display(income_preference_gap)


gender_age_delta_rows = []

gender_age_target = gender_age_metric_compare[
    gender_age_metric_compare["데이터"].isin(["valid", "test"])
].copy()

for model_name in gender_age_target["모델"].unique():
    for data_name in ["valid", "test"]:
        group_keys = (
            gender_age_target[
                (gender_age_target["모델"] == model_name)
                & (gender_age_target["데이터"] == data_name)
            ][["성별_라벨", "연령대"]]
            .drop_duplicates()
        )
        
        for _, key_row in group_keys.iterrows():
            sex_value = key_row["성별_라벨"]
            age_value = key_row["연령대"]
            
            base_row = gender_age_target[
                (gender_age_target["실험"] == "baseline")
                & (gender_age_target["모델"] == model_name)
                & (gender_age_target["데이터"] == data_name)
                & (gender_age_target["성별_라벨"] == sex_value)
                & (gender_age_target["연령대"] == age_value)
            ]
            income_row = gender_age_target[
                (gender_age_target["실험"] == "income")
                & (gender_age_target["모델"] == model_name)
                & (gender_age_target["데이터"] == data_name)
                & (gender_age_target["성별_라벨"] == sex_value)
                & (gender_age_target["연령대"] == age_value)
            ]
            
            if len(base_row) == 0 or len(income_row) == 0:
                continue
            
            base_row = base_row.iloc[0]
            income_row = income_row.iloc[0]
            
            gender_age_delta_rows.append({
                "모델": model_name,
                "데이터": data_name,
                "성별_라벨": sex_value,
                "연령대": age_value,
                "응답자수": income_row["응답자수"],
                "Top1_Accuracy_변화": income_row["Top1_Accuracy"] - base_row["Top1_Accuracy"],
                "Top3_HitRate_변화": income_row["Top3_HitRate"] - base_row["Top3_HitRate"],
                "MRR_변화": income_row["MRR"] - base_row["MRR"],
                "NDCG_3_변화": income_row["NDCG_3"] - base_row["NDCG_3"],
                "Rank1_LogLoss_변화": income_row["Rank1_LogLoss"] - base_row["Rank1_LogLoss"],
            })

gender_age_delta = pd.DataFrame(gender_age_delta_rows).round(4)

print("성연령별 소득 변수 추가 효과")
display(gender_age_delta.sort_values(["데이터", "모델", "Top1_Accuracy_변화"], ascending=[True, True, False]))

print("income 모델 성연령별 Top1 하위 그룹 - valid / 다항로지스틱")
display(
    gender_age_metric_compare[
        (gender_age_metric_compare["실험"] == "income")
        & (gender_age_metric_compare["모델"] == "다항로지스틱")
        & (gender_age_metric_compare["데이터"] == "valid")
    ]
    .sort_values("Top1_Accuracy")
    .head(10)
)

print("income 모델 성연령별 Top1 상위 그룹 - valid / 다항로지스틱")
display(
    gender_age_metric_compare[
        (gender_age_metric_compare["실험"] == "income")
        & (gender_age_metric_compare["모델"] == "다항로지스틱")
        & (gender_age_metric_compare["데이터"] == "valid")
    ]
    .sort_values("Top1_Accuracy", ascending=False)
    .head(10)
)

model_performance_compare.to_csv(
    INCOME_EXP_PATH / f"{OUTPUT_PREFIX}_income_exp_model_performance.csv",
    index=False,
    encoding="utf-8-sig",
)

classification_metric_compare.to_csv(
    INCOME_EXP_PATH / f"{OUTPUT_PREFIX}_income_exp_classification_metric.csv",
    index=False,
    encoding="utf-8-sig",
)

category_metric_compare.to_csv(
    INCOME_EXP_PATH / f"{OUTPUT_PREFIX}_income_exp_category_metric.csv",
    index=False,
    encoding="utf-8-sig",
)

income_delta.to_csv(
    INCOME_EXP_PATH / f"{OUTPUT_PREFIX}_income_exp_performance_delta.csv",
    index=False,
    encoding="utf-8-sig",
)


gender_age_metric_compare.to_csv(
    INCOME_EXP_PATH / f"{OUTPUT_PREFIX}_income_exp_gender_age_metric.csv",
    index=False,
    encoding="utf-8-sig",
)

gender_age_delta.to_csv(
    INCOME_EXP_PATH / f"{OUTPUT_PREFIX}_income_exp_gender_age_delta.csv",
    index=False,
    encoding="utf-8-sig",
)

print("저장 완료")
print(INCOME_EXP_PATH / f"{OUTPUT_PREFIX}_income_exp_model_performance.csv")
print(INCOME_EXP_PATH / f"{OUTPUT_PREFIX}_income_exp_performance_delta.csv")
